# LoRA fine-tune of Qwen2.5-0.5B as the cascade router / intent classifier
**NOT RUN IN THIS ENVIRONMENT** (needs a GPU). Written to be run by you on a free Colab T4.

- Runtime: ~20-30 min (train ~12 min on 4k examples, 1 epoch, plus eval and GGUF export).
- Steps: LoRA SFT on the Bitext **train split** (labels = 27 intents; 80/10/10 split, seed 0) -> evaluate on the **held-out split**
  vs the prompt-only (zero-shot) 0.5B router -> merge + export to GGUF so the gateway can load it.
- Output: `results/router_lora.json` (accuracy numbers) is downloaded at the end; commit it into `results/`.
- Optional stretch: a short GRPO run (TRL) rewarding valid tool-call JSON and the correct tool (cell guarded by `RUN_GRPO`).

In [ ]:
!pip -q install transformers datasets peft trl accelerate bitsandbytes pandas
import os, json, random, urllib.request
os.makedirs("results", exist_ok=True)
urllib.request.urlretrieve("https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/resolve/main/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv", "bitext.csv")
import pandas as pd
df = pd.read_csv("bitext.csv").sample(frac=1, random_state=0).reset_index(drop=True)
n = len(df); train, val, test = df[: int(.8 * n)], df[int(.8 * n): int(.9 * n)], df[int(.9 * n):]
INTENTS = sorted(df.intent.unique()); print(len(INTENTS), "intents;", len(train), len(val), len(test))

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = "left"
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).cuda()
PROMPT = "Classify the customer message into one intent.\nIntents: " + ", ".join(INTENTS) + "\nMessage: {m}\nIntent:"

def predict(model, msgs, bs=32):
    out = []
    for i in range(0, len(msgs), bs):
        enc = tok([PROMPT.format(m=m) for m in msgs[i:i+bs]], return_tensors="pt", padding=True).to("cuda")
        g = model.generate(**enc, max_new_tokens=8, do_sample=False)
        out += [tok.decode(x[enc.input_ids.shape[1]:], skip_special_tokens=True).strip().split("\n")[0] for x in g]
    return out

sample = test.sample(500, random_state=0)
zs = predict(base, sample.instruction.tolist())
zs_acc = sum(p == t for p, t in zip(zs, sample.intent)) / len(sample); print("prompt-only router accuracy:", zs_acc)

In [ ]:
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
tr = train.sample(4000, random_state=0)
ds = Dataset.from_dict({"text": [PROMPT.format(m=m) + " " + i + tok.eos_token for m, i in zip(tr.instruction, tr.intent)]})
model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM"))
SFTTrainer(model=model, train_dataset=ds, processing_class=tok,
           args=SFTConfig(output_dir="lora_out", num_train_epochs=1, per_device_train_batch_size=16, learning_rate=2e-4,
                          logging_steps=20, fp16=True, report_to=[], dataset_text_field="text", max_length=256)).train()

In [ ]:
model.eval(); ft = predict(model, sample.instruction.tolist())
ft_acc = sum(p == t for p, t in zip(ft, sample.intent)) / len(sample); print("LoRA router accuracy:", ft_acc)
json.dump({"model": MODEL, "n_eval": len(sample), "split": "held-out 10% (seed 0)", "prompt_only_accuracy": zs_acc,
           "lora_accuracy": ft_acc, "train_examples": 4000, "epochs": 1}, open("results/router_lora.json", "w"), indent=1)

In [ ]:
# Merge + export to GGUF (llama.cpp converter), then quantize to Q4_K_M like the gateway's replicas
merged = model.merge_and_unload(); merged.save_pretrained("merged"); tok.save_pretrained("merged")
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp && pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py merged --outfile router-f16.gguf --outtype f16
!cd llama.cpp && cmake -B build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF >/dev/null && cmake --build build -j --target llama-quantize >/dev/null
!llama.cpp/build/bin/llama-quantize router-f16.gguf router-q4_k_m.gguf Q4_K_M
from google.colab import files; files.download("router-q4_k_m.gguf"); files.download("results/router_lora.json")

In [ ]:
RUN_GRPO = False   # stretch goal: set True to run a short GRPO job (~10 min). Reward: valid tool-call JSON + correct tool.
if RUN_GRPO:
    import re
    from trl import GRPOTrainer, GRPOConfig
    TOOLS = {"get_order_details", "cancel_pending_order", "find_user_id_by_email"}
    def reward(completions, gold_tool, **kw):
        r = []
        for c, g in zip(completions, gold_tool):
            try: o = json.loads(re.search(r"\{.*\}", c, re.S).group(0)); ok = o["tool"] in TOOLS and isinstance(o["args"], dict)
            except Exception: ok = False
            r.append(0.5 * ok + 0.5 * (ok and o["tool"] == g))
        return r
    prompts = [{"prompt": f"Customer: cancel my order #W{1000+i}. Reply with JSON {{\"tool\": ..., \"args\": {{...}}}}.", "gold_tool": "cancel_pending_order"} for i in range(64)]
    GRPOTrainer(model=MODEL, reward_funcs=reward, train_dataset=Dataset.from_list(prompts),
                args=GRPOConfig(output_dir="grpo_out", max_steps=30, per_device_train_batch_size=8, num_generations=4, report_to=[])).train()